In [ ]:
import pandas as pd

df = pd.read_excel("cities20.xlsx", index_col=0)

In [ ]:
# Percentilisek számítása - BŐVÍTETT VERZIÓ

def calculate_percentiles(df):
    percentiles = {}
    
    # Földrajzi percentilek
    geo_cols = [col for col in df.columns if '_per_100k' in col and 'geo_' in col]
    for col in geo_cols:
        df[f'{col}_percentile'] = df[col].rank(pct=True)
        percentiles[col.replace('geo_', '').replace('_per_100k', '')] = df[f'{col}_percentile'].to_dict()
    
    # Életstílus percentilek
    lifestyle_cols = [col for col in df.columns if '_per_100k' in col and 'eletstilus_' in col]
    for col in lifestyle_cols:
        df[f'{col}_percentile'] = df[col].rank(pct=True)
        percentiles[col.replace('eletstilus_', '').replace('_per_100k', '')] = df[f'{col}_percentile'].to_dict()
    
    # Turista sűrűség percentil
    if 'crowding_per_100k' in df.columns:
        df['crowding_percentile'] = df['crowding_per_100k'].rank(pct=True)
        percentiles['crowding'] = df['crowding_percentile'].to_dict()
    
    # Ár percentilis - Numbeo adatok alapján
    price_cols = [col for col in df.columns if col.startswith('col_') and pd.api.types.is_numeric_dtype(df[col])]
    if price_cols:
        # Átlagos ár index számítása (egyszerűsítve)
        df['avg_price_index'] = df[price_cols].mean(axis=1, skipna=True)
        df['price_percentile'] = 1- df['avg_price_index'].rank(pct=True)
        percentiles['price'] = df['price_percentile'].to_dict()
    
    # Klíma percentilis - átlaghőmérséklet alapján
    climate_cols = [col for col in df.columns if col.startswith('climate_temp_mean_')]
    if climate_cols:
        # Éves átlaghőmérséklet számítása
        df['annual_avg_temp'] = df[climate_cols].mean(axis=1, skipna=True)
        df['climate_percentile'] = df['annual_avg_temp'].rank(pct=True)
        percentiles['climate'] = df['climate_percentile'].to_dict()
    
    # Távolság percentilis
    if 'distance' in df.columns:
        df['distance_percentile'] = df['distance'].rank(pct=True)
        # Fordítva, mert a kisebb távolság jobb
        df['distance_percentile'] = 1 - df['distance_percentile']
        percentiles['distance'] = df['distance_percentile'].to_dict()
    
    return percentiles

# Percentilek számítása
percentile_data = calculate_percentiles(df)

# Cities dictionary frissítése percentilis értékekkel
cities = {}
for city in df.index:
    cities[city] = {
        "földrajz": {
            "tengerpart": percentile_data.get('beach', {}).get(city, 0.5),
            "hegy": percentile_data.get('mountain', {}).get(city, 0.5),
            "város": 0.7,  # placeholder vagy más metrika
            "sziget": percentile_data.get('island', {}).get(city, 0.5),
            "tópart": percentile_data.get('lake', {}).get(city, 0.5),
            "sivatag": percentile_data.get('desert', {}).get(city, 0.1)
        },
        "ár": percentile_data.get('price', {}).get(city, 0.5),
        "klíma": percentile_data.get('climate', {}).get(city, 0.5),
        "életstílus": {
            "bulis": percentile_data.get('bulis', {}).get(city, 0.5),
            "relax": percentile_data.get('relax', {}).get(city, 0.5),
            "aktív": 0.5,  # placeholder
            "kulturális": percentile_data.get('kulturalis', {}).get(city, 0.5),
            "családbarát": percentile_data.get('csaladbarat', {}).get(city, 0.5)
        },
        "távolság": percentile_data.get('distance', {}).get(city, 0.5),
        "zsúfoltság": percentile_data.get('crowding', {}).get(city, 0.5)
    }

# Ellenőrzés - első néhány város adatainak megjelenítése
for city in list(df.index)[:3]:
    print(f"\n{city} percentilis értékek:")
    print(f"  Ár: {cities[city]['ár']:.3f}")
    print(f"  Klíma: {cities[city]['klíma']:.3f}")
    print(f"  Távolság: {cities[city]['távolság']:.3f}")

In [ ]:
# User input (csúszkák, multi-select)
user_preferences = {
    "súlyok": {  # nulladik kérdés: mennyire fontos az egyes dimenzió
        "földrajz": 8,
        "ár": 9,
        "klíma": 7,
        "életstílus": 10,
        "távolság": 6,
        "zsúfoltság": 5,
    },
    "földrajz": {"tengerpart": 8, "hegy": 3, "város":5, "sziget":7, "tópart":2, "sivatag":1},
    "ár": 0.9,         
    "klíma": 0.8,      
    "életstílus": {"bulis":4, "relax":8, "aktív":6, "kulturális":7, "családbarát":5},
    "távolság": 0.9,   
    "zsúfoltság": 0.7,
}


# Normalizált súlyok (összeg=1)
total_weight = sum(user_preferences["súlyok"].values())
weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}

# Simple similarity function
def similarity(user_val, city_val):
    return 1 - abs(user_val - city_val)

# Weighted similarity per city
def city_score(user_pref, city_data, weights):
    total = 0
    for attr, weight in weights.items():
        if attr in ["földrajz", "életstílus"]:
            # átlag a multi-select értékekből
            user_vals = user_pref[attr]
            city_vals = city_data[attr]
            sim_vals = []
            for k in user_vals.keys():
                sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
            avg_sim = sum(sim_vals)/len(sim_vals)
            total += weight * avg_sim
        else:
            # EGYÉNI érték
            user_val = user_pref[attr]
            city_val = city_data[attr]
            sim = similarity(user_val, city_val)
            total += weight * sim
    return total


# Calculate scores for all cities
city_scores = {}
for city_name, city_data in cities.items():
    score = city_score(user_preferences, city_data, weights)
    city_scores[city_name] = score

# Sort top cities
top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)

# Output
print("Top ajánlott városok a preferenciáid alapján:")
for city, score in top_cities:
    print(f"{city}: {score:.3f}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# Interaktív súlyok és preferenciák beállítása
def create_interactive_preferences():
    # Fő súlyok
    weight_sliders = {
        "földrajz": widgets.IntSlider(value=8, min=1, max=10, description='Földrajz:', style={'description_width': 'initial'}),
        "ár": widgets.IntSlider(value=9, min=1, max=10, description='Ár:', style={'description_width': 'initial'}),
        "klíma": widgets.IntSlider(value=7, min=1, max=10, description='Klíma:', style={'description_width': 'initial'}),
        "életstílus": widgets.IntSlider(value=10, min=1, max=10, description='Életstílus:', style={'description_width': 'initial'}),
        "távolság": widgets.IntSlider(value=6, min=1, max=10, description='Távolság:', style={'description_width': 'initial'}),
        "zsúfoltság": widgets.IntSlider(value=5, min=1, max=10, description='Zsúfoltság:', style={'description_width': 'initial'})
    }
    
    # Földrajzi preferenciák
    geo_sliders = {
        "tengerpart": widgets.IntSlider(value=8, min=1, max=10, description='Tengerpart:', style={'description_width': 'initial'}),
        "hegy": widgets.IntSlider(value=3, min=1, max=10, description='Hegy:', style={'description_width': 'initial'}),
        "város": widgets.IntSlider(value=5, min=1, max=10, description='Város:', style={'description_width': 'initial'}),
        "sziget": widgets.IntSlider(value=7, min=1, max=10, description='Sziget:', style={'description_width': 'initial'}),
        "tópart": widgets.IntSlider(value=2, min=1, max=10, description='Tópart:', style={'description_width': 'initial'}),
        "sivatag": widgets.IntSlider(value=1, min=1, max=10, description='Sivatag:', style={'description_width': 'initial'})
    }
    
    # Életstílus preferenciák
    lifestyle_sliders = {
        "bulis": widgets.IntSlider(value=4, min=1, max=10, description='Bulis:', style={'description_width': 'initial'}),
        "relax": widgets.IntSlider(value=8, min=1, max=10, description='Relax:', style={'description_width': 'initial'}),
        "aktív": widgets.IntSlider(value=6, min=1, max=10, description='Aktív:', style={'description_width': 'initial'}),
        "kulturális": widgets.IntSlider(value=7, min=1, max=10, description='Kulturális:', style={'description_width': 'initial'}),
        "családbarát": widgets.IntSlider(value=5, min=1, max=10, description='Családbarát:', style={'description_width': 'initial'})
    }
    
    # Egyéni preferenciák
    individual_sliders = {
        "ár": widgets.FloatSlider(value=0.9, min=0, max=1, step=0.1, description='Ár preferencia:', style={'description_width': 'initial'}),
        "klíma": widgets.FloatSlider(value=0.8, min=0, max=1, step=0.1, description='Klíma preferencia:', style={'description_width': 'initial'}),
        "távolság": widgets.FloatSlider(value=0.9, min=0, max=1, step=0.1, description='Távolság preferencia:', style={'description_width': 'initial'}),
        "zsúfoltság": widgets.FloatSlider(value=0.7, min=0, max=1, step=0.1, description='Zsúfoltság preferencia:', style={'description_width': 'initial'})
    }
    
    # Gomb a számításhoz
    calculate_button = widgets.Button(description="Ajánlások megjelenítése", button_style='success')
    
    # Layout
    weight_box = widgets.VBox([weight_sliders[k] for k in weight_sliders])
    geo_box = widgets.VBox([geo_sliders[k] for k in geo_sliders])
    lifestyle_box = widgets.VBox([lifestyle_sliders[k] for k in lifestyle_sliders])
    individual_box = widgets.VBox([individual_sliders[k] for k in individual_sliders])
    
    tab = widgets.Tab()
    tab.children = [weight_box, geo_box, lifestyle_box, individual_box]
    tab.titles = ['Súlyok', 'Földrajz', 'Életstílus', 'Egyéni preferenciák']
    
    output = widgets.Output()
    
    def on_calculate_click(b):
        with output:
            clear_output()
            
            # Adatok gyűjtése
            user_preferences = {
                "súlyok": {k: v.value for k, v in weight_sliders.items()},
                "földrajz": {k: v.value for k, v in geo_sliders.items()},
                "ár": individual_sliders["ár"].value,
                "klíma": individual_sliders["klíma"].value,
                "életstílus": {k: v.value for k, v in lifestyle_sliders.items()},
                "távolság": individual_sliders["távolság"].value,
                "zsúfoltság": individual_sliders["zsúfoltság"].value,
            }
            
            # Normalizált súlyok
            total_weight = sum(user_preferences["súlyok"].values())
            weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}
            
            # Hasonlóság számítás
            def similarity(user_val, city_val):
                return 1 - abs(user_val - city_val)
            
            def city_score(user_pref, city_data, weights):
                total = 0
                for attr, weight in weights.items():
                    if attr in ["földrajz", "életstílus"]:
                        user_vals = user_pref[attr]
                        city_vals = city_data[attr]
                        sim_vals = []
                        for k in user_vals.keys():
                            sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
                        avg_sim = sum(sim_vals)/len(sim_vals)
                        total += weight * avg_sim
                    else:
                        user_val = user_pref[attr]
                        city_val = city_data[attr]
                        sim = similarity(user_val, city_val)
                        total += weight * sim
                return total
            
            # Pontszámok számítása
            city_scores = {}
            for city_name, city_data in cities.items():
                score = city_score(user_preferences, city_data, weights)
                city_scores[city_name] = score
            
            # Top városok
            top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)[:10]
            
            # Eredmények megjelenítése
            print("🏆 Top 10 ajánlott városok a preferenciáid alapján:")
            print("-" * 50)
            for i, (city, score) in enumerate(top_cities, 1):
                print(f"{i}. {city}: {score:.3f}")
            
            # Részletesebb információk az első 3 városról
            print("\n" + "="*50)
            print("📊 Részletes információk a top 3 városról:")
            for i, (city, score) in enumerate(top_cities[:3], 1):
                print(f"\n{i}. {city}:")
                city_data = cities[city]
                print(f"   • Földrajz: {city_data['földrajz']}")
                print(f"   • Ár: {city_data['ár']:.3f}")
                print(f"   • Klíma: {city_data['klíma']:.3f}")
                print(f"   • Távolság: {city_data['távolság']:.3f}")
    
    calculate_button.on_click(on_calculate_click)
    
    display(widgets.VBox([tab, calculate_button, output]))

# Interaktív felület indítása
create_interactive_preferences()